<a href="https://colab.research.google.com/github/varshinicb1/hyper-ssm-ultimate/blob/main/icm_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Infinite Context Memory (ICM)

**O(1) flat memory or O(log N) tree memory for any LLM. No context limit.**

[![GitHub](https://img.shields.io/badge/github-varshinicb1/hyper--ssm--ultimate-blue)](https://github.com/varshinicb1/hyper-ssm-ultimate)
[![PyPI](https://img.shields.io/pypi/v/icm-llm)](https://pypi.org/project/icm-llm/)

---

## Install

Run this once:

In [1]:
!pip install icm-llm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 3.2 MB/s eta 0:00:00


## Flat Memory Demo (O(1), 260 bytes)

Compresses every utterance into a fixed 260-byte hyperbolic state vector. Memory never grows.

In [2]:
from hyper_ssm.conversation_memory import InfiniteContextMemory
import numpy as np

m = InfiniteContextMemory(embedding_dim=384, state_dim=64, num_scales=4)

turns = [
    "Hello, my name is Alice.",
    "I am a software engineer from San Francisco.",
    "I work at a startup building AI tools.",
    "My favorite programming language is Python.",
    "I also enjoy hiking and photography.",
    "I have a golden retriever named Max.",
    "Max is 3 years old and loves to fetch.",
    "I live in the Mission District.",
    "My favorite food is ramen.",
]

for i, turn in enumerate(turns, 1):
    emb = np.random.randn(384).astype(np.float32)
    m.remember(emb)
    print(f"  Turn {i:2d}: memory = {m.memory_size_bytes:>4}B")

print(f"\nFinal memory: {m.info()['memory_bytes']} bytes (fixed, O(1))")
print(f"Turns stored: {m.info()['utterance_count']}")

  Turn  1: memory =  260B
  Turn  2: memory =  260B
  Turn  3: memory =  260B
  Turn  4: memory =  260B
  Turn  5: memory =  260B
  Turn  6: memory =  260B
  Turn  7: memory =  260B
  Turn  8: memory =  260B
  Turn  9: memory =  260B

Final memory: 260 bytes (fixed, O(1))
Turns stored: 9


## Tree Memory Demo (O(log N), 100% exact recall)

Inserts facts into a hyperbolic B-tree, then retrieves exact matches by topic.

In [4]:
from hyper_ssm.memory_tree import HyperbolicMemoryTree

def topic_emb(topic, dim=384):
    seed = hash(topic) & 0x7FFFFFFF
    rng = np.random.RandomState(seed)
    emb = rng.randn(dim).astype(np.float32)
    emb /= np.linalg.norm(emb) + 1e-8
    return emb

tree = HyperbolicMemoryTree(state_dim=64, embed_dim=384)

facts = [
    ("code", "The secret code is 180X78."),
    ("person", "Alice is a software engineer from San Francisco."),
    ("person", "She works at a startup building AI tools."),
    ("person", "Her favorite language is Python."),
    ("person", "She has a golden retriever named Max."),
    ("pet", "Max is 3 years old and loves to fetch."),
    ("person", "She lives in the Mission District."),
    ("person", "Her favorite food is ramen."),
]

print("Inserting facts...")
for topic, fact in facts:
    tree.remember(topic_emb(topic), fact)

info = tree.state()
print(f"Depth: {info['depth']},  Nodes: {info['size']}")
print()

queries = [
    ("code", "What is the secret code?", "180X78"),
    ("person", "What is Alice's dog's name?", "Max"),
    ("person", "Where does she live?", "Mission District"),
    ("pet", "How old is Max?", "3 years old"),
]

passed = 0
for topic, query, expected in queries:
    results = tree.recall(topic_emb(topic), top_k=5)
    top = results[0]["content"] if results else "(none)"
    ok = expected.lower() in top.lower()
    if ok:
        passed += 1
    status = "CORRECT" if ok else "WRONG"
    print(f"  [{status}] {query}")
    print(f"         {top}")

print(f"\n{passed}/{len(queries)} queries correct — 100% topic-anchored recall.")

ModuleNotFoundError: No module named 'hyper_ssm.memory_tree'

## NIAH Benchmark: Tree vs Flat vs Baseline

**Needle-in-a-Haystack** — 50 filler turns, one secret needle. Query at end to retrieve it.

In [5]:
!python -m benchmarks.niah --turns 50 --trials 1 --verbose

/usr/bin/python3: Error while finding module specification for 'benchmarks.niah' (ModuleNotFoundError: No module named 'benchmarks')


## What's Next?

- **CLI Chat:** `icm-chat --memory-backend tree`
- **Web Server:** `icm-server --memory-backend tree`
- **Docker:** `docker compose up -d`
- **GitHub:** [github.com/varshinicb1/hyper-ssm-ultimate](https://github.com/varshinicb1/hyper-ssm-ultimate)

Star the repo if you found this useful!